In [1]:
# 1- setup
import os, hashlib, shutil
from pathlib import Path
import numpy as np, pandas as pd, imagehash
from PIL import Image, ImageOps, ImageDraw
from tqdm.auto import tqdm

# move to the project root no matter where this notebook lives
root = Path.cwd()
while not (root / "data" / "imgs_raw").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("working in:", Path.cwd())

RAW = Path("data/imgs_cleaned")
OUT = Path("logs/dedup_audit")
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
MIN_SIDE = 100      # smaller than this = too tiny
THRESH = 10          # max hash distance (of 64); BOTH hashes must agree
CHROMA_MAX = 0.2   # max colour-balance difference (brightness-proof)

def load(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)      # respect phone rotation tags
    return img.convert("RGB")

TRANSFORMS = [None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180,
              Image.Transpose.ROTATE_270, Image.Transpose.FLIP_LEFT_RIGHT,
              Image.Transpose.FLIP_TOP_BOTTOM, Image.Transpose.TRANSPOSE,
              Image.Transpose.TRANSVERSE]

working in: c:\Projects\University\Sri-Lankan-Food-Recognition-and-Information-System


In [2]:
# 2-fingerprint every img
rows, hp, hd, chroma = [], [], [], []
files = sorted(p for p in RAW.rglob("*") if p.suffix.lower() in EXTS)
for p in tqdm(files):
    parts = p.relative_to(RAW).parts
    rec = dict(path=str(p), cls=parts[0], source=parts[1] if len(parts) > 2 else "original")
    try:
        img = load(p)
        small = img.copy(); small.thumbnail((256, 256))
        g = ImageOps.autocontrast(small.convert("L"))     # cancels brightness changes
        gts = [g if t is None else g.transpose(t) for t in TRANSFORMS]
        a_ = [int(str(imagehash.phash(x)), 16) for x in gts]
        b_ = [int(str(imagehash.dhash(x)), 16) for x in gts]
        px = np.asarray(small.resize((32, 32))).reshape(-1, 3).astype(float).mean(0)
        c_ = px / max(px.sum(), 1e-6)                     # colour balance, ignores brightness
        rec.update(ok=True, w=img.size[0], h=img.size[1], bytes=p.stat().st_size,
                   sha=hashlib.sha256(p.read_bytes()).hexdigest())
    except Exception as e:
        rec.update(ok=False, error=str(e)); a_ = b_ = [0] * 8; c_ = np.zeros(3)
    rows.append(rec); hp.append(a_); hd.append(b_); chroma.append(c_)

df = pd.DataFrame(rows)
HP = np.array(hp, dtype=np.uint64); HD = np.array(hd, dtype=np.uint64)
CH = np.array(chroma, dtype=np.float32)
print(df.groupby("cls").size(), "\nunreadable:", (~df.ok).sum())

  0%|          | 0/1371 [00:00<?, ?it/s]

cls
dosa              181
hoppers           156
idli               86
kottu              66
milk_rice         143
paratha           114
pittu             249
rice              293
string_hoppers     83
dtype: int64 
unreadable: 0


In [3]:
# 3-find same-photo pairs and group them
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    x = np.bitwise_xor.outer(a, b)
    return POP[x.view(np.uint8).reshape(*x.shape, 8)].sum(axis=-1).astype(np.int16)

mask = df.ok.values
d = df[mask].reset_index(drop=True)
d[["w", "h", "bytes"]] = d[["w", "h", "bytes"]].astype(int)
HPd, HDd, CHd = HP[mask], HD[mask], CH[mask]

D = np.full((len(d), len(d)), 64, dtype=np.int16)
for v in range(8):                                   # all 8 orientations; both hashes must agree
    D = np.minimum(D, np.maximum(hamming(HPd[:, 0], HPd[:, v]), hamming(HDd[:, 0], HDd[:, v])))
C = np.zeros((len(d), len(d)), dtype=np.float32)     # colour-balance difference
for k in range(3):
    C += np.abs(CHd[:, k][:, None] - CHd[:, k][None, :])
same = (D <= THRESH) & (C <= CHROMA_MAX)

parent = list(range(len(d)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i, j in zip(*np.where(np.triu(same, k=1))):
    parent[find(i)] = find(j)

# also group Roboflow exports by original name (idli18_jpg.rf.xxxx -> idli18_jpg)
first = {}
for i, p in enumerate(d.path):
    name = Path(p).name
    if ".rf." not in name:
        continue
    k = (Path(p).parent.as_posix(), name.split(".rf.")[0])   # same folder + same original
    if k in first:
        parent[find(i)] = find(first[k])
    else:
        first[k] = i

d["group"] = [find(i) for i in range(len(d))]
d["gsize"] = d.groupby("group").group.transform("size")
print("10 biggest groups:", [int(x) for x in sorted(d.gsize.unique())[-10:]])
print("groups with copies:", d[d.gsize > 1].group.nunique())
print(d[d.gsize > 1].groupby("cls").group.nunique().rename("groups per class"))

10 biggest groups: [1, 2, 3, 4]
groups with copies: 37
cls
dosa          1
hoppers       7
milk_rice     7
pittu        10
rice         13
Name: groups per class, dtype: int64


In [4]:
# 4-suggest keepers + build review sheets
shutil.rmtree(OUT / "sheets", ignore_errors=True); (OUT / "sheets").mkdir(parents=True)
dups = d[d.gsize > 1].copy()
dups["group"] = dups.group.rank(method="dense").astype(int)
dups["area"] = dups.w * dups.h
dups = dups.sort_values(["group", "area", "bytes"], ascending=[True, False, False])
dups["keep"] = ~dups.duplicated("group")             # largest image in each group
dups["cross_class"] = dups.groupby("group").cls.transform("nunique") > 1
dups.loc[dups.cross_class, "keep"] = False           # label conflict -> you decide
dups[["group", "keep", "cross_class", "cls", "source", "w", "h", "path"]].to_csv(OUT / "dup_groups.csv", index=False)

def make_sheet(g, out_path, tile=240):
    items = g.head(12).to_dict("records")
    cols = min(4, len(items)); n = -(-len(items) // cols)
    canvas = Image.new("RGB", (cols * tile, n * (tile + 30)), "white"); dr = ImageDraw.Draw(canvas)
    for k, r in enumerate(items):
        im = load(r["path"]); im.thumbnail((tile, tile))
        x, y = (k % cols) * tile, (k // cols) * (tile + 30)
        canvas.paste(im, (x, y))
        dr.text((x + 4, y + tile + 2), f'{"KEEP" if r["keep"] else "drop"} | {r["cls"]} | {r["w"]}x{r["h"]}', fill="black")
        dr.text((x + 4, y + tile + 14), Path(r["path"]).name[:34], fill="black")
    canvas.save(out_path, quality=85)

for gid, g in tqdm(list(dups.groupby("group"))):
    tag = "conflict" if g.cross_class.iloc[0] else "dup"
    make_sheet(g, OUT / "sheets" / f"{tag}_group_{gid:04d}.jpg")
print("review sheets in", OUT / "sheets")

  0%|          | 0/37 [00:00<?, ?it/s]

review sheets in logs\dedup_audit\sheets
